# Moosic — 01. Baseline Clustering

**Convention used throughout this project's notebooks:**
- Main pipeline steps are numbered `1, 2, 3, ...` — run these in order, top to bottom.
- Experimental variants on a step (different feature sets, different k, different scalers) are labelled `5a`, `5b`, etc. — run whichever variant you're curious about, they don't depend on each other.


## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn import set_config
import os

set_config(transform_output="pandas")
os.makedirs("../outputs", exist_ok=True)
RANDOM_STATE = 42

## 2. Load Data

In [ ]:
df = pd.read_csv("../data/5000_songs.csv")
df.columns = df.columns.str.strip()

print(f"Shape: {df.shape}")
df.head()

**Note:** dataset shape should be checked here before going further — if this doesn't say roughly (5000, ~19), something upstream (wrong file, duplicate rows) needs investigating before continuing.


## 3. Feature Selection & Correlation Check

The case study's example code used 5 features (`danceability, energy, acousticness, tempo, valence`) — this was the course's illustrative starting point, not a data-driven choice. Before trusting it, check correlations across a wider candidate set to spot redundant features and confirm which additions actually carry new signal.


In [ ]:
candidate_features = ['danceability', 'energy', 'acousticness', 'tempo', 'valence',
                       'loudness', 'speechiness', 'instrumentalness', 'liveness']

corr = df[candidate_features].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', center=0, vmin=-1, vmax=1)
plt.title("Feature correlation matrix (candidate set)")
plt.tight_layout()
plt.savefig("../outputs/correlation_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

**Findings from this check (see project report for full discussion):**
- `loudness` correlates strongly (~0.8+) with `energy` and `acousticness` — redundant, dropped.
- `speechiness` and `instrumentalness` correlate weakly with everything else — genuine new signal, kept.
- `key` (cyclic, 0–11) and `mode` (binary) need special handling to be usable at all — excluded for now, noted as future work.
- `time_signature` and `duration_ms` carry little information for mood/style — excluded.

**Final feature set used going forward:**


In [ ]:
features = ['danceability', 'energy', 'acousticness', 'tempo', 'valence',
            'speechiness', 'instrumentalness']
features

## 4. Scaling

In [ ]:
scaler = MinMaxScaler()
scaled = scaler.fit_transform(df[features])

scaled.describe().round(3)

## 5. Choosing k (elbow + silhouette)

In [ ]:
k_values = range(2, 20)
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].plot(k_values, inertias, 'bo-')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Inertia'); axes[0].set_title('Elbow Method'); axes[0].grid(True)
axes[1].plot(k_values, silhouettes, 'ro-')
axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhouette Score'); axes[1].set_title('Silhouette Score'); axes[1].grid(True)
plt.tight_layout()
plt.savefig("../outputs/01_elbow_silhouette.png", dpi=150, bbox_inches="tight")
plt.show()

### 5a. Experiment — same k range, raw 5-feature set (for comparison against the trimmed 7)

Run this cell to compare against Step 5's main result using the *original*, unexamined 5-feature list — useful for showing in the report whether trimming/adding features actually changed anything meaningful.

Uses its own `scaler_5a`, separate from the main pipeline's `scaler` — safe to run in any order, won't affect Steps 6/7.


In [ ]:
features_5a = ['danceability', 'energy', 'acousticness', 'tempo', 'valence']
scaler_5a = MinMaxScaler()  # own scaler — never touches the main pipeline's `scaler`
scaled_5a = scaler_5a.fit_transform(df[features_5a])

silhouettes_5a = []
for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(scaled_5a)
    silhouettes_5a.append(silhouette_score(scaled_5a, labels))

comparison_df = pd.DataFrame({
    "k": list(k_values),
    "silhouette_7feat": silhouettes,
    "silhouette_5feat_original": silhouettes_5a
})
comparison_df

### 5b. Experiment — a narrower k range (2–10) matching the course notebook's default

Useful if you want to sanity-check against the exact range shown in the course material, rather than the wider 2–20 used above.


In [ ]:
k_values_5b = range(2, 11)
silhouettes_5b = [
    silhouette_score(scaled, KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10).fit_predict(scaled))
    for k in k_values_5b
]
pd.DataFrame({"k": list(k_values_5b), "silhouette": silhouettes_5b})

## 6. Baseline Clustering at k=8

k=8 was chosen from the silhouette knee in Step 5, on the full 5000-song dataset, with **no target playlist size imposed yet** — this is deliberately the "free" clustering, used to see what the data naturally looks like before any business constraint is applied (see notebook `02_recursive_pipeline.ipynb` for the constrained version).


In [ ]:
BEST_K = 8

kmeans = KMeans(n_clusters=BEST_K, random_state=RANDOM_STATE, n_init=10)
df['cluster'] = kmeans.fit_predict(scaled)

print(df['cluster'].value_counts().sort_index())

## 7. Radar Chart of Cluster Profiles

**Important:** plot the *scaled* centroids directly (already 0–1 under `MinMaxScaler`), not inverse-transformed ones — inverse-transforming back to original units (e.g. BPM) and then forcing a `(0, 1)` axis was the bug that made an earlier version of this chart show all clusters overlapping. Keep the inverse-transformed version only for the printed reference table below.


In [ ]:
# scaled centroids — for the chart
scaled_centroid_df = pd.DataFrame(kmeans.cluster_centers_, columns=features)

# original-unit centroids — for the readable table only
original_centroids = scaler.inverse_transform(kmeans.cluster_centers_)
centroid_df = pd.DataFrame(original_centroids, columns=features)
centroid_df['cluster'] = range(BEST_K)

print("=== Cluster centroids (original scale, for reading) ===")
print(centroid_df.round(3))

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(projection='polar'))
angles = np.linspace(0, 2 * np.pi, len(features), endpoint=False).tolist()
angles += angles[:1]

for i in range(BEST_K):
    values = scaled_centroid_df.iloc[i][features].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=f'Cluster {i}')
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(features)
ax.set_ylim(0, 1)
ax.set_title(f'Cluster Profiles (K={BEST_K})', size=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
plt.tight_layout()
plt.savefig("../outputs/01_radar_k8.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Save Outputs for the Next Notebook

`02_recursive_pipeline.ipynb` starts fresh from the CSV (doesn't depend on this notebook's in-memory variables), but saving the labelled baseline here keeps a record of what "free" clustering looked like before any business constraint was applied.


In [ ]:
df.to_csv("../outputs/01_baseline_clustered.csv", index=False)
print("Saved to ../outputs/01_baseline_clustered.csv")

---
**Next notebook:** `02_recursive_pipeline.ipynb` — introduces the target playlist size constraint and the recursive split / reassign / merge pipeline.
